# Automated Evaluation of Document Understanding Models for Document to JSON (v2)

This notebook uses the refactored `utils.evaluation` package for evaluation.

- **Multi-metric assessment** (Exact Match, Character Error Rate, ROUGE)
- **Feature type analysis**
- **Interactive visualizations**
- **Model performance comparison**

# Setup

In [ ]:
# TODO needs fixing - see utils.helpers
!rm -rf ./data/results/results

In [ ]:
%pip install Levenshtein==0.26.1 plotnine==0.14.5 itables==2.2.4 regex==2024.11.6 evaluate==0.4.3 cer==1.2.0 rouge_score==0.1.2 seaborn sagemaker==2.256.1 boto3 botocore --quiet

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itables import init_notebook_mode, show
import itables.options as opt
opt.columnDefs = [{"width": "100px", "targets": "_all"}]
opt.column_filters = "footer"
opt.showIndex = False

# Data Loading

In [ ]:
import sagemaker
import boto3
import os
import pandas as pd

region = "us-east-1"
boto_session = boto3.Session(region_name=region)
session = sagemaker.Session(boto_session=boto_session)
default_bucket_name = session.default_bucket()

dataset_s3_prefix = "fatura2-train-data-amazon-nova-v5-300-no-table"
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"
dataset_base_dir = f"./data/processed/{dataset_s3_prefix}/"

print("Downloading dataset...")
!aws s3 cp $dataset_s3_uri $dataset_base_dir --recursive --quiet

In [ ]:
from utils.evaluation import ResultsRegistry

registry = ResultsRegistry()
registry.add_from_csv('./results_to_compare.csv', dataset_base_dir)

df_test = registry.load_all()
print(f"Loaded {len(df_test)} samples from {df_test['name'].nunique()} models")
df_test['name'].value_counts()

## View Sample

In [ ]:
from IPython.display import JSON
JSON(df_test.iloc[0].to_dict(), root="inference sample")

# Edit Distance Computation

In [ ]:
from utils.evaluation.metrics import compute_edit_distance

text_property_name = None

# Get all entities
entities = set()
for _, row in df_test.iterrows():
    if isinstance(row['labels'], dict):
        entities.update(row['labels'].keys())
entities = sorted(list(entities))

df_dist = compute_edit_distance(df_test, entities, text_property_name)
df_dist = pd.concat([df_test.reset_index(drop=True), df_dist[entities]], axis=1)
print(f"Computed edit distance for {len(entities)} entities")

# Feature Type Analysis and Categorization

## Analyse Feature Distribution of Entities

In [ ]:
from utils.evaluation.categorization import analyze_feature_distribution, categorize_features, FeatureCategory

# Use first model for feature analysis (ground truth is the same across models)
df_single = df_test[df_test['name'] == df_test['name'].iloc[0]]

stats_df = pd.DataFrame([analyze_feature_distribution(df_single, e) for e in entities])
stats_df.head()

## Show Entities by Missing GroundTruth

In [ ]:
from utils.evaluation.visualizations import plot_null_percentage_by_entity

plot_null_percentage_by_entity(stats_df)

## Entity String Length Analysis with Boxplots

Analyze and categorize entities based on their string lengths using boxplots.

**Feature Categorization**

1. **Missing Ground Truth**: Entities with high null percentage
2. **Short Text**: Entities with shorter length (e.g., dates, amounts)
3. **Long Text**: Entities with longer, free-form text

In [ ]:
from utils.evaluation.visualizations import plot_length_boxplots

plot_length_boxplots(df_single)

## Apply Feature Categorization

In [ ]:
feature_categories = categorize_features(df_test)

cat_df = pd.DataFrame.from_dict(feature_categories, orient='index', columns=['category'])
cat_df['category'] = cat_df['category'].astype(str)
with pd.option_context('display.max_rows', None):
    for name, group in cat_df.groupby('category'):
        print(f"\nGroup: {name}")
        print(group)

In [ ]:
# Filter entities with valid ground truth
entities_to_keep = [k for k, v in feature_categories.items() if v != FeatureCategory.MISSING_GROUND_TRUTH]
print(f"Entities with valid ground truth: {len(entities_to_keep)}")

# Edit Distance Heatmap

In [ ]:
import matplotlib.pyplot as plt
from utils.evaluation.visualizations import plot_edit_distance_heatmap

# Sort entities by category, then alphabetically within each category
entity_cat_df = pd.DataFrame([
    {'entity': e, 'entity_type': str(feature_categories.get(e, FeatureCategory.MISSING_GROUND_TRUTH))}
    for e in entities
]).sort_values(['entity_type', 'entity']).reset_index(drop=True)

ordered_all = entity_cat_df['entity'].tolist()
change_indices = entity_cat_df.index[entity_cat_df['entity_type'] != entity_cat_df['entity_type'].shift()].tolist()
vlines_all = [idx + 0.5 for idx in change_indices[1:]]

### All Entities

In [ ]:
ed_df_all = df_dist[['file_id', 'name'] + entities].copy()
plot_edit_distance_heatmap(ed_df_all, ordered_all, width=15, height=10, geom_vlines=vlines_all)

### Relevant Entities (without missing ground truth)

In [ ]:
# Filter to entities with valid ground truth and recompute category separator lines
filtered_cat_df = entity_cat_df[entity_cat_df['entity_type'] != str(FeatureCategory.MISSING_GROUND_TRUTH)].reset_index(drop=True)
ordered_filtered = filtered_cat_df['entity'].tolist()
change_indices_filtered = filtered_cat_df.index[filtered_cat_df['entity_type'] != filtered_cat_df['entity_type'].shift()].tolist()
vlines_filtered = [idx + 0.5 for idx in change_indices_filtered[1:]]

ed_df_filtered = df_dist[['file_id', 'name'] + ordered_filtered].copy()
plot_edit_distance_heatmap(ed_df_filtered, ordered_filtered, width=15, height=10, geom_vlines=vlines_filtered)

# Compute Metrics

In [ ]:
from utils.evaluation.metrics import compute_exact_match, compute_cer, compute_rouge

em_results, cer_results, rouge_results = [], [], []

for model_name in df_test['name'].unique():
    df_model = df_test[df_test['name'] == model_name]
    
    em_df = compute_exact_match(df_model, entities_to_keep)
    em_df['model'] = model_name
    em_results.append(em_df)
    
    cer_df = compute_cer(df_model, entities_to_keep, text_property_name)
    cer_df['model'] = model_name
    cer_results.append(cer_df)
    
    rouge_df = compute_rouge(df_model, entities_to_keep, text_property_name)
    rouge_df['model'] = model_name
    rouge_results.append(rouge_df)

em_all = pd.concat(em_results)
cer_all = pd.concat(cer_results)
rouge_all = pd.concat(rouge_results)

In [ ]:
print("Exact Match by Entity:")
em_pivot = em_all.pivot(index='entity', columns='model', values='exact_match')
display(em_pivot)

In [ ]:
print("CER by Entity:")
cer_pivot = cer_all.pivot(index='entity', columns='model', values='cer_score')
display(cer_pivot)

# Metric Visualizations

In [ ]:
from utils.evaluation.visualizations import (
    plot_exact_match_comparison,
    plot_metric_distribution,
    plot_category_performance,
    plot_null_statistics,
)

## Exact Match Comparison

In [ ]:
plot = plot_exact_match_comparison(em_all)
plot

## CER Distribution

In [ ]:
plot = plot_metric_distribution(cer_all, 'cer_score')
plot

## Performance by Feature Category

In [ ]:
em_cer = em_all.merge(cer_all, on=['entity', 'model'])

plot = plot_category_performance(em_cer, feature_categories, metric='exact_match')
plot

## Null Statistics

In [ ]:
from utils.evaluation.metrics import compute_null_statistics

# Compute per model and concat
null_results = []
for model_name in df_test['name'].unique():
    df_model = df_test[df_test['name'] == model_name]
    null_df = compute_null_statistics(df_model, entities, model=model_name)
    null_results.append(null_df)
null_all = pd.concat(null_results)

plot = plot_null_statistics(null_all, metric='null_in_predictions_pct')
plot

In [ ]:
plot = plot_null_statistics(null_all, metric='missing_key_pct')
plot

# Model Comparison

In [ ]:
from utils.evaluation import EvaluationConfig, MultiModelComparator

config = EvaluationConfig(
    compute_exact_match=True,
    compute_edit_distance=True,
    compute_cer=True,
    compute_rouge=True
)

results_dict = {name: df_test[df_test['name'] == name].copy() for name in df_test['name'].unique()}

comparator = MultiModelComparator(results_dict, config)
comparator.compare()

print("Model Comparison:")
display(comparator.get_comparison_table())

In [ ]:
print("Model Rankings (1 = best):")
display(comparator.get_ranking())

In [ ]:
from utils.evaluation.visualizations import plot_model_ranking

fig = plot_model_ranking(comparator.get_comparison_table())
fig

# Summary Table

In [ ]:
summary_data = []
for model_name in df_test['name'].unique():
    summary_data.append({
        'model': model_name,
        'accuracy (exact match)': round(em_all[em_all['model'] == model_name]['exact_match'].mean(), 2),
        'cer_score': round(cer_all[cer_all['model'] == model_name]['cer_score'].mean(), 5),
        'rouge1': round(rouge_all[rouge_all['model'] == model_name]['rouge1'].mean(), 2),
        'rouge2': round(rouge_all[rouge_all['model'] == model_name]['rouge2'].mean(), 2),
        'rougeL': round(rouge_all[rouge_all['model'] == model_name]['rougeL'].mean(), 2)
    })

summary_df = pd.DataFrame(summary_data)
print("Aggregated Model Comparison:")
display(summary_df)

# Export Results

In [ ]:
# from utils.evaluation.reports import export_results

# os.makedirs('./output', exist_ok=True)
# export_results(summary_df, './output/model_summary.csv', format='csv')
# export_results(em_pivot, './output/exact_match_by_entity.csv', format='csv')
# export_results(cer_pivot, './output/cer_by_entity.csv', format='csv')
# print("Results exported to ./output/")